# Notebook 01 – Environment Exploration

Dieses Notebook dokumentiert die Unity-Environments, die für das Reinforcement-Learning-Projekt erstellt wurden.

Ziel dieses Notebooks ist nicht das Training eines Agenten, sondern die nachvollziehbare Beschreibung des experimentellen Setups. Die Environments bilden die Grundlage für alle späteren Trainings- und Evaluationsergebnisse. Deshalb wird hier festgehalten, wie die Aufgabe aufgebaut ist, welche Rolle die einzelnen Environments haben und welche Designentscheidungen für Agent, Action Space, Observation Space und Reward-Struktur getroffen wurden.

---

## 1. Überblick über die Environments

Im Projekt wurden drei Unity-Environments erstellt:

| Environment | Rolle im Projekt | Beschreibung |
|---|---|---|
| Env1_Simple | Trainings- und Hauptvergleichsumgebung | Einfacheres Maze zur grundlegenden Navigation |
| Env2_Complex | Transferumgebung | Komplexere Maze-Struktur zur Prüfung von Generalisierung |
| Env3_Warehouse | Transferumgebung | Lagerhausähnliche Umgebung als praxisnähere Transferaufgabe |

Env1 dient als zentrale Trainings- und Evaluationsumgebung. Hier werden die Algorithmen primär trainiert und miteinander verglichen. Env2 und Env3 werden anschließend genutzt, um zu prüfen, ob gelernte Policies auf neue Layouts übertragen werden können.

---

## 2. Env1 – Simple Maze

![Env1 Simple](../docs/images/env1_simple.png)

Env1 ist die primäre Trainings- und Vergleichsumgebung. Der Agent startet an einem Spawn Point und soll das Goal erreichen, ohne gegen Walls zu laufen.

Diese Umgebung ist bewusst einfacher gehalten als die späteren Transferumgebungen. Dadurch kann zunächst geprüft werden, ob die Algorithmen grundsätzlich zielgerichtetes Navigationsverhalten lernen können.

---

## 3. Env2 – Complex Maze

![Env2 Complex](../docs/images/env2_complex.png)

Env2 ist eine komplexere Transferumgebung. Sie wurde nicht primär zum Training verwendet, sondern dient zur Prüfung, ob ein in Env1 trainierter Agent auf eine neue Maze-Struktur generalisieren kann.

Die Umgebung ist anspruchsvoller als Env1 und soll zeigen, ob das gelernte Verhalten nur auf das ursprüngliche Layout passt oder zumindest teilweise auf andere Strukturen übertragbar ist.

---

## 4. Env3 – Warehouse

![Env3 Warehouse](../docs/images/env3_warehouse.png)

Env3 ist als warehouse-artige Transferumgebung gestaltet. Sie soll prüfen, ob das gelernte Navigationsverhalten über ein klassisches Maze hinaus auf eine praxisnähere Struktur übertragen werden kann.

Damit erweitert Env3 das Experiment um eine Umgebung, die stärker an ein mögliches Anwendungsszenario erinnert.

---

## 5. Agent, Goal und Walls

Der Agent bewegt sich innerhalb der Unity-Szene und muss das Goal erreichen. Die wichtigsten Objekte sind:

| Objekt | Funktion |
|---|---|
| Agent | Lernt die Navigation durch das Environment |
| Goal | Zielobjekt, das erreicht werden soll |
| Walls | Hindernisse, die vermieden werden müssen |
| Spawn Points | Startpositionen für Episoden |
| Floor | Bewegungsfläche des Agenten |

Ein erfolgreiches Verhalten bedeutet, dass der Agent das Goal erreicht, ohne vorher gegen eine Wall zu laufen oder bis zum Episodenlimit im Environment zu bleiben.

---

## 6. Action Space

Der Agent nutzt einen diskreten Action Space mit 9 Aktionen.

Die Aktionen kombinieren Bewegungs- und Rotationsentscheidungen:

| Action Index | Bewegung / Rotation |
|---:|---|
| 0 | No-op |
| 1 | Forward |
| 2 | Backward |
| 3 | Turn Left |
| 4 | Forward + Turn Left |
| 5 | Backward + Turn Left |
| 6 | Turn Right |
| 7 | Forward + Turn Right |
| 8 | Backward + Turn Right |

Diese Struktur wurde gewählt, weil der Agent nicht wie in einer klassischen Grid World feldweise springen soll. Stattdessen bewegt er sich in Unity kontinuierlicher durch den Raum, während die Entscheidungen für die Algorithmen trotzdem diskret bleiben.

Dadurch unterscheidet sich das Setup z. B. von einfachen Grid-World-Umgebungen wie FrozenLake.

---

## 7. Observation Space

Die Python-Evaluation zeigt für die Unity-Umgebung einen Observation Space der Form:

```text
Box(-inf, inf, (48,), float32)
```

Der Agent erhält damit eine vektorbasierte Beobachtung der Umgebung. Diese Beobachtungen bilden die Grundlage für die Policy-Entscheidungen der Algorithmen.

Die Beobachtungsstruktur wurde über die Experimente hinweg konstant gehalten, damit A2C, DQN und PPO auf derselben Navigationsaufgabe arbeiten.

---

## 8. Reward Design

Die finale Reward-Struktur folgt einem einfachen Navigationsziel:

- Goal erreichen: hoher positiver Reward
- Wall-Hit: deutlicher negativer Reward
- Jeder Step: kleine negative Step Penalty
- Timeout: Episode endet ohne Goal

Die finale Logik war:

```text
Goal:    +10 - 0.0005 * steps
Wall:    -5  - 0.0005 * steps
Timeout:      -0.0005 * steps
```

Dieses Reward Design soll den Agenten dazu bringen, das Goal möglichst schnell zu erreichen und Wall-Hits zu vermeiden.

Gleichzeitig entsteht dadurch ein Sparse-Reward-Problem: Der wichtigste positive Reward wird nur beim tatsächlichen Erreichen des Goals vergeben. Gerade zu Beginn des Trainings passiert das selten. Dadurch erhält der Agent nur wenige positive Lernsignale und lernt überwiegend aus negativen Signalen wie Wall-Hits, Timeouts und Step Penalty.

Dieses Problem ist eine zentrale Herausforderung des Projekts und beeinflusst die späteren Ergebnisse stark.

---

## 9. Rolle der Environments im Experiment

Die Environments erfüllen unterschiedliche Rollen:

- Env1 wird für Training und Hauptvergleich genutzt.
- Env2 prüft Generalisierung auf ein komplexeres Maze.
- Env3 prüft Transfer auf eine warehouse-artige Struktur.

Damit wird nicht nur geprüft, ob ein Agent in einer einzelnen Umgebung funktioniert, sondern ob gelernte Navigation zumindest teilweise auf neue Layouts übertragen werden kann.

---

## 10. Fazit

Dieses Notebook dokumentiert das Environment-Design als Grundlage des gesamten Projekts.

Die späteren Trainingsergebnisse müssen vor diesem Setup interpretiert werden: Die Aufgabe ist nicht nur algorithmisch, sondern auch stark durch Environment Design, Action Space, Observation Space und Reward Design geprägt.

Gerade die später beobachteten Probleme wie Lazy-Agent-Verhalten, Wall-Fails, Timeouts und eingeschränkte Transferfähigkeit hängen eng mit diesen Designentscheidungen zusammen.